# Final Project

**Name:** -- Federico Garcia Rodriguez --

**e-mail:** -- federico.garcia0747@alumnos.udg.mx --

**code:** -- 224807479 --

# Modulo

In [638]:
import numpy as np
import pandas as pd
import math

import panel as pn
pn.extension()
import panel.widgets as pnw

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

pn.extension('plotly')

# Dataframe

In [639]:
data = pd.read_csv("FinalProject\heart_attack_prediction_dataset.csv")
data.head()

#Frequency table
age=data["Age"]
age_table = pd.crosstab(data['Age'], "frequency") 
cholesterol_table = pd.crosstab(data['Cholesterol'], "frequency") 
heart_rate_table = pd.crosstab(data['Heart Rate'], "frequency")
triglycerides_table = pd.crosstab(data['Triglycerides'], "frequency")

# DASHBOARD

## Widgets

In [648]:
#labels
str_pane1 = pn.pane.Str(
    'Variable',
    styles={'font-size': '12pt'}
)
str_pane2 = pn.pane.Str(
    'Parameters',
    styles={'font-size': '12pt'}
)
str_pane3 = pn.pane.Str(
    'Metrics',
    styles={'font-size': '12pt'}
)
str_pane4 = pn.pane.Str(
    'Graphic',
    styles={'font-size': '12pt'}
)
str_pane5 = pn.pane.Str(
    'Metrics',
    styles={'font-size': '12pt'}
)

#RW options
radio_group = pnw.RadioButtonGroup(name='RB_parameter_options', options=['Frequency table', 'Prediction'], value='Frequency table', button_type='light')

#Metrics selector
Metrics_selector = pnw.Select(name='Metrics type', options=['Age', 'Cholesterol', 'Heart Rate', 'Triglycerides', 'All'], value='Age')

## Metrics

In [656]:
input_age = pnw.IntInput(name='Age', value=18, step=1, start=18, end=90)
input_cholesterol = pnw.IntInput(name='Cholesterol', value=120, step=10, start=120, end=400)
input_triglycerides = pnw.FloatInput(name='Triglycerides', value=30, step=50, start=30, end=800)
input_heart_rate = pnw.FloatInput(name='Heart Rate', value=40, step=5, start=40, end=110)

In [657]:
#column paremeters
column_metrics = pn.Column(input_age, input_cholesterol, input_triglycerides, input_heart_rate)
column_parameters = pn.Column(Metrics_selector)

#column layout
@pn.depends(radio_group)
def param_select(radio_group):
    column=""
    match radio_group:
        case 'Frequency table': column = column_parameters
        case 'Prediction': column = column_metrics
    return column

@pn.depends(radio_group, Metrics_selector)
def plot_select(radio_group, Metrics_selector):
    plot_selected=""
    match Metrics_selector:
        case 'Age': plot_selected = plot_age
        case 'Cholesterol': plot_selected = plot_cholesterol
        case 'Heart Rate': plot_selected = plot_heart_rate
        case 'Triglycerides': plot_selected = plot_triglycerides
        case 'All': plot_selected = plot_all
            
    return plot_selected


# Decorators

## Graphics

In [658]:
def plot_age():
    fig = px.scatter(age_table, x=["frequency"], marginal_x="histogram", marginal_y="rug")
    return fig

def plot_cholesterol():
    fig = px.density_heatmap(cholesterol_table, x=["frequency"], marginal_x="box", marginal_y="box")
    return fig

def plot_triglycerides():
    fig = px.histogram(triglycerides_table, x=["frequency"], marginal="box")
    return fig

def plot_heart_rate():
    fig = px.scatter(heart_rate_table, x=["frequency"], marginal_x="box")
    return fig

def plot_all():
    fig = make_subplots(rows=2, cols=2)
    
    hist1 =  go.Histogram(x=age_table['frequency'], name="Age")
    hist2 =  go.Histogram(x=cholesterol_table['frequency'], name="Cholesterol")
    hist3 =  go.Histogram(x=triglycerides_table['frequency'], name="Triglycerides")
    hist4 =  go.Histogram(x=heart_rate_table["frequency"], name="Heart Rate")

    fig.append_trace(hist1, 1, 1)
    fig.append_trace(hist2, 1, 2)
    fig.append_trace(hist3, 2, 1)
    fig.append_trace(hist4, 2, 2)

    fig.update_layout(title="Frequencies distribution")

    return fig

## Deploy Dashboard

In [659]:
column_params = pn.Column(str_pane1, radio_group, str_pane3, param_select, styles=dict(background='WhiteSmoke'))
column_graph = pn.Column(str_pane4, plot_select)
column_stat = pn.Column(str_pane4, plot_select)

if radio_group=="Frequency table":
    row = pn.Row(column_params, column_graph, styles=dict(background='WhiteSmoke'))
else:
    row = pn.Row(column_params, column_stat, styles=dict(background='WhiteSmoke'))
    
server=row.show()

Launching server at http://localhost:53961
